<a href="https://colab.research.google.com/github/AkshatMishra29/Innomatics_Internship_Generative_Ai-_module/blob/main/LangChain_Deep_Technical_Blog_codes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🦜🔗 LangChain Deep Technical Blog — Complete Code Notebook

**Assignment:** GenAI – LangChain Deep Technical Blog  
**Internship:** Data Science Internship – February 2026  

---

## Table of Contents
1. [Setup & Installation](#setup)
2. [Basic LLM Call](#llm)
3. [Prompt Templates](#prompts)
4. [Chains (LCEL)](#chains)
5. [Memory](#memory)
6. [Agents & Tools](#agents)
7. [Document Loaders](#loaders)
8. [Vector Stores (Indexes)](#vectorstores)
9. [End-to-End RAG Pipeline](#rag)

---


---
## 🔧 Section 1 — Setup & Installation <a name='setup'></a>

In [7]:
# ============================================================
# STEP 1: Install all required libraries
# Including langchain-google-genai for Gemini support
# ============================================================

!pip install -q langchain langchain-google-genai langchain-community \
                langchain-core faiss-cpu tiktoken \
                pypdf python-dotenv duckduckgo-search

In [8]:
# ============================================================
# STEP 2: Set your Google API Key (Gemini)
# Method A — Using Colab Secrets (Recommended)
# Go to 🔑 Secrets (left sidebar) → Add GOOGLE_API_KEY
# ============================================================

import os
from google.colab import userdata

try:
    os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_API_KEY")
    print("✅ Gemini API key loaded from Colab Secrets")
except Exception:
    os.environ["GOOGLE_API_KEY"] = "your-gemini-api-key-here"
    print("⚠️ API key set manually")

✅ Gemini API key loaded from Colab Secrets


In [9]:
# ============================================================
# STEP 3: Verify installation and imports
# ============================================================

import langchain
import openai

print(f"✅ LangChain version : {langchain.__version__}")
print(f"✅ OpenAI version    : {openai.__version__}")
print("✅ All imports successful — ready to go!")

✅ LangChain version : 1.2.14
✅ OpenAI version    : 2.30.0
✅ All imports successful — ready to go!


---
## 🤖 Section 2 — Basic LLM Call <a name='llm'></a>

### Concept
The **LLM / Chat Model** is the brain of any LangChain application.  
LangChain wraps raw API calls into a unified interface so you can swap
models (OpenAI, HuggingFace, Anthropic) without changing your pipeline code.

| Parameter | Meaning |
|---|---|
| `model` | Which GPT model to use |
| `temperature` | 0 = deterministic, 1 = creative |
| `max_tokens` | Max length of the response |

In [10]:
# ============================================================
# Basic LLM Call using ChatGoogleGenerativeAI
# ============================================================

from langchain_google_genai import ChatGoogleGenerativeAI

# Initialize the Gemini Model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7,
    max_output_tokens=300
)

response = llm.invoke("What is LangChain and why is it useful?")

print("📤 Response:")
print(response.content)

📤 Response:
**What is LangChain?**

LangChain is an


In [11]:
# ============================================================
# Using HumanMessage / SystemMessage for role-based prompting
# ============================================================

from langchain_core.messages import HumanMessage, SystemMessage

# System message sets the assistant's behavior/role
# Human message is the actual user input
messages = [
    SystemMessage(content="You are a concise Python tutor. Keep answers under 100 words."),
    HumanMessage(content="What is a list comprehension in Python?")
]

response = llm.invoke(messages)
print("📤 Response:")
print(response.content)

📤 Response:
A list comprehension provides a concise way to create lists in Python. It's often a more readable and efficient alternative to using a `for` loop with `append`.

Its basic syntax is


---
## 📝 Section 3 — Prompt Templates <a name='prompts'></a>

### Concept
**Prompt Templates** let you create reusable, parameterized prompts.  
Instead of hardcoding questions, you define a template with `{variables}`
and fill them in at runtime — making your code **modular and reusable**.

In [13]:
# ============================================================
# PromptTemplate — for simple string prompts
# ============================================================

# Updated import for LangChain v0.2+
from langchain_core.prompts import PromptTemplate

# Define a template with {placeholders}
template = PromptTemplate(
    input_variables=["topic", "level"],
    template="Explain {topic} to a {level} student in 3 bullet points."
)

# Format fills in the variables
formatted_prompt = template.format(topic="neural networks", level="beginner")
print("📝 Formatted Prompt:")
print(formatted_prompt)

# Send it to the LLM
response = llm.invoke(formatted_prompt)
print("\n📤 Response:")
print(response.content)

📝 Formatted Prompt:
Explain neural networks to a beginner student in 3 bullet points.

📤 Response:
Here are three bullet points explaining neural networks to a beginner


In [15]:
# ============================================================
# ChatPromptTemplate — for multi-role conversations
# Best practice for Chat Models (GPT-3.5, GPT-4)
# ============================================================

from langchain_core.prompts import ChatPromptTemplate

# Define system + user messages as a template
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert in {domain}. Be precise and technical."),
    ("human",  "Explain {concept} with a real-world analogy.")
])

# format_messages() returns a list of message objects
messages = chat_prompt.format_messages(
    domain="Data Science",
    concept="gradient descent"
)

print("📝 Formatted Messages:")
for msg in messages:
    print(f"  [{msg.__class__.__name__}]: {msg.content}")

response = llm.invoke(messages)
print("\n📤 Response:")
print(response.content)

📝 Formatted Messages:
  [SystemMessage]: You are an expert in Data Science. Be precise and technical.
  [HumanMessage]: Explain gradient descent with a real-world analogy.

📤 Response:
Gradient Descent (GD) is an iterative optimization algorithm used to


---
## ⛓️ Section 4 — Chains (LCEL) <a name='chains'></a>

### Concept
**Chains** connect components into a pipeline using the **`|` (pipe) operator**,
known as LangChain Expression Language (LCEL).  

```
Prompt Template  →  LLM  →  Output Parser  →  Final Result
```

Each component's output becomes the next component's input automatically.

In [18]:
# ============================================================
# Simple Chain: Prompt → LLM → Output Parser
# ============================================================

from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser

# 1. Define prompt template
prompt = ChatPromptTemplate.from_template(
    "Summarize the concept of {topic} in exactly 2 sentences."
)

# 2. Define the LLM (Using a verified model name)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)


# 3. Output parser converts AIMessage → plain string
parser = StrOutputParser()

# 4. Chain them together using LCEL pipe operator |
chain = prompt | llm | parser

# 5. Run the chain
result = chain.invoke({"topic": "transformer architecture"})

print("📤 Chain Output:")
print(result)

📤 Chain Output:
The transformer architecture revolutionized sequence modeling by employing self-attention mechanisms to weigh the importance of different parts of an input sequence, capturing long-range dependencies effectively. This design allows for highly parallelizable computation, significantly speeding up training and inference compared to recurrent neural networks, making it dominant in natural language processing and other sequence-to-sequence tasks.


In [19]:
# ============================================================
# Sequential Chain — Output of Chain 1 feeds into Chain 2
# Use case: Generate a concept, then generate quiz from it
# ============================================================

from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Initialize Gemini
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.5)
parser = StrOutputParser()

# Chain 1: Explain the topic
explain_prompt = ChatPromptTemplate.from_template(
    "Explain {topic} briefly in 3 sentences."
)

# Chain 2: Create a quiz based on explanation
quiz_prompt = ChatPromptTemplate.from_template(
    "Based on this explanation:\n{explanation}\n\nWrite 2 MCQ quiz questions."
)

# Build the sequential pipeline
# Step 1 → explanation, Step 2 → quiz from explanation
chain1 = explain_prompt | llm | parser

full_chain = (
    {"explanation": chain1, "topic": RunnablePassthrough()}
    | quiz_prompt
    | llm
    | parser
)

result = full_chain.invoke({"topic": "backpropagation"})
print("📤 Quiz Output:")
print(result)

📤 Quiz Output:
Here are two MCQ quiz questions based on the explanation provided:

---

**Question 1:**
According to the explanation, what is the primary function of the Backpropagation algorithm?
A) To randomly assign initial values to network weights.
B) To efficiently calculate the gradients of the loss function concerning the network's weights.
C) To determine the optimal number of hidden layers in a neural network.
D) To activate the neurons in the input layer during the forward pass.

**Correct Answer:** B

---

**Question 2:**
Which mathematical rule is explicitly mentioned as being used by Backpropagation to propagate error backward through the network, layer by layer?
A) The product rule of calculus.
B) The quotient rule of calculus.
C) The chain rule of calculus.
D) The power rule of calculus.

**Correct Answer:** C


---
## 🧠 Section 5 — Memory <a name='memory'></a>

### Concept
By default, LLMs are **stateless** — they forget every previous message.
**Memory** solves this by storing conversation history and injecting it
into every new prompt, enabling multi-turn conversations.

| Memory Type | What it stores |
|---|---|
| `ConversationBufferMemory` | Full history (all messages) |
| `ConversationSummaryMemory` | Summarized history (saves tokens) |
| `ConversationWindowMemory` | Only last N turns |

In [27]:
# Ensure the latest core and genai packages are installed
!pip install -q --upgrade langchain-core langchain-google-genai

import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

# 1. Initialize the Gemini Model
# Make sure your GOOGLE_API_KEY environment variable is set
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.5)

# 2. Create a Prompt Template with a history placeholder
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful and concise AI assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

# 3. Combine prompt and model into a chain (LCEL syntax)
chain = prompt | llm

# 4. Set up a dictionary to store memory for different sessions
store = {}

def get_session_history(session_id: str) -> InMemoryChatMessageHistory:
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

# 5. Wrap the chain with message history (The modern "Buffer Memory")
conversational_chain = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

# ============================================================
# Execution
# ============================================================

# We must pass a session_id so the system knows which memory buffer to use
config = {"configurable": {"session_id": "session_1"}}

# Turn 1
print("=" * 50)
print("Turn 1:")
response1 = conversational_chain.invoke(
    {"input": "Hi! My name is Rahul and I'm learning LangChain."},
    config=config
)
print(f"Bot: {response1.content}")

# Turn 2 — bot should remember the name
print("\n" + "=" * 50)
print("Turn 2:")
response2 = conversational_chain.invoke(
    {"input": "What's my name and what am I learning?"},
    config=config
)
print(f"Bot: {response2.content}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.7/508.7 kB 7.8 MB/s eta 0:00:00
Turn 1:
Bot: Hi Rahul! That's great to hear. LangChain is a powerful library.

How can I help you with your learning today?

Turn 2:
Bot: Your name is Rahul, and you're learning LangChain.


In [29]:
# ============================================================
# Inspect what's stored in memory
# FIXED: Updated to work with the 'store' dictionary used above
# ============================================================

print("📦 Memory Contents (Session 1):")
if "session_1" in store:
    history = store["session_1"].messages
    for msg in history:
        print(f"[{type(msg).__name__}]: {msg.content}")
else:
    print("No history found for session_1.")

📦 Memory Contents (Session 1):
[HumanMessage]: Hi! My name is Rahul and I'm learning LangChain.
[AIMessage]: Hi Rahul! That's great to hear. LangChain is a powerful library.

How can I help you with your learning today?
[HumanMessage]: What's my name and what am I learning?
[AIMessage]: Your name is Rahul, and you're learning LangChain.


In [59]:
# Ensure the latest core and genai packages are installed
!pip install -q --upgrade langchain-core langchain-google-genai

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. Initialize the Gemini Model
# (Using gemini-1.5-flash as 2.5 is not a standard accessible model identifier yet)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# 2. Create the modern summarization prompt
# We explicitly tell the LLM how to manage the summary state
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "Progressively summarize the lines of conversation provided. Add onto the previous summary to create a new, concise summary. Return ONLY the updated summary text."),
    ("user", "Current summary:\n{current_summary}\n\nNew interaction:\nHuman: {human_input}\nAI: {ai_output}")
])

# 3. Build the LCEL Chain (Prompt -> LLM -> String)
summarizer_chain = summary_prompt | llm | StrOutputParser()

# ============================================================
# Execution & State Management
# ============================================================

# In modern LangChain, you manage the state explicitly
current_summary = ""

# Simulate saving context (Turn 1)
print("Processing Turn 1...")
current_summary = summarizer_chain.invoke({
    "current_summary": current_summary,
    "human_input": "I am building a chatbot for a hospital.",
    "ai_output": "Great! Medical chatbots require careful accuracy."
})

# Simulate saving context (Turn 2)
print("Processing Turn 2...")
current_summary = summarizer_chain.invoke({
    "current_summary": current_summary,
    "human_input": "It should answer FAQs about appointments.",
    "ai_output": "Use RAG with hospital policy documents for best results."
})

# Output the final summarized memory
print("\n📦 Modern Summary Memory State:")
print(current_summary)

Processing Turn 1...
Processing Turn 2...

📦 Modern Summary Memory State:
A user is building a hospital chatbot that needs to answer FAQs about appointments. The AI emphasizes the need for careful accuracy and suggests using RAG with hospital policy documents for best results.


---
## 🕵️ Section 6 — Agents & Tools <a name='agents'></a>

### Concept
A **Chain** follows a fixed sequence of steps.  
An **Agent** is dynamic — it uses the LLM as a **reasoning engine** to decide:
- Which tool to use
- When to use it
- What to do with the result

The agent follows a **ReAct loop**: Thought → Action → Observation → repeat until final answer.

In [33]:
# ============================================================
# Custom Tool — Define your own tools for the agent
# ============================================================

from langchain.tools import tool

# @tool decorator turns any Python function into a LangChain tool
@tool
def calculate_bmi(weight_kg: float, height_m: float) -> str:
    """Calculates BMI given weight in kilograms and height in meters."""
    bmi = weight_kg / (height_m ** 2)
    if bmi < 18.5:
        category = "Underweight"
    elif bmi < 25:
        category = "Normal weight"
    elif bmi < 30:
        category = "Overweight"
    else:
        category = "Obese"
    return f"BMI: {bmi:.2f} — Category: {category}"


@tool
def word_count(text: str) -> str:
    """Counts the number of words in a given text string."""
    count = len(text.split())
    return f"Word count: {count} words"


# Test tools directly
print(calculate_bmi.invoke({"weight_kg": 70, "height_m": 1.75}))
print(word_count.invoke({"text": "LangChain is a framework for building LLM applications"}))

BMI: 22.86 — Category: Normal weight
Word count: 8 words


In [48]:
# Ensure you have langgraph installed alongside the core packages
!pip install -q --upgrade langchain-core langchain-google-genai langgraph

from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.prebuilt import create_react_agent

# ============================================================
# 1. Define Tools (CRITICAL FIX)
# You MUST use @tool, type hints (float, str), and docstrings!
# ============================================================

@tool
def calculate_bmi(weight_kg: float, height_m: float) -> float:
    """Calculates the Body Mass Index (BMI) given weight in kg and height in meters."""
    return weight_kg / (height_m ** 2)

@tool
def word_count(text: str) -> int:
    """Counts the number of words in a given text string."""
    return len(text.split())

tools = [calculate_bmi, word_count]

# ============================================================
# 2. Initialize Gemini
# ============================================================
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# ============================================================
# 3. Create the Modern LangGraph Agent
# This replaces AgentExecutor and all the prompt boilerplate!
# ============================================================
agent_executor = create_react_agent(llm, tools)

# ============================================================
# 4. Execution
# LangGraph expects input as a list of messages
# ============================================================

print("=" * 50)
print("🤖 Agent Execution:")

result = agent_executor.invoke({
    "messages": [("user", "What is the BMI of a person who weighs 65 kg and is 1.70 m tall?")]
})

# LangGraph returns the entire conversational state.
# The final answer is simply the content of the last message in the list.
print("\n📤 Final Answer:")
print(result["messages"][-1].content)

/tmp/ipykernel_6231/2507796671.py:34: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_executor = create_react_agent(llm, tools)


🤖 Agent Execution:

📤 Final Answer:
The BMI of the person is 22.49.


---
## 📄 Section 7 — Document Loaders <a name='loaders'></a>

### Concept
**Document Loaders** ingest data from external sources (text files, PDFs, websites,
databases) and convert them into LangChain `Document` objects with `.page_content`
and `.metadata` fields — ready to be processed by the rest of your pipeline.

In [49]:
# ============================================================
# Create a sample document to load (simulating a real file)
# ============================================================

# Create a sample text file to use as our document
sample_text = """LangChain Architecture Overview

LangChain is an open-source framework developed to simplify building applications
powered by large language models (LLMs). It provides a modular set of components
that developers can combine to create sophisticated LLM pipelines.

Core Components:
1. Models: Wrappers around LLMs and chat models
2. Prompts: Templates for structuring model inputs
3. Chains: Sequences of component calls
4. Memory: State persistence across chain calls
5. Agents: Dynamic decision-making using tools
6. Tools: External functions agents can call
7. Document Loaders: Data ingestion from various sources
8. Vector Stores: Semantic similarity search

LangChain enables developers to rapidly prototype and deploy LLM applications
by abstracting the complexity of working with models, data, and tools.
"""

with open("sample_doc.txt", "w") as f:
    f.write(sample_text)

print("✅ Sample document created: sample_doc.txt")

✅ Sample document created: sample_doc.txt


In [39]:
# ============================================================
# TextLoader — Load plain text files
# ============================================================

from langchain_community.document_loaders import TextLoader

# Load the file
loader    = TextLoader("sample_doc.txt")
documents = loader.load()

print(f"📂 Number of documents loaded : {len(documents)}")
print(f"📝 Content preview            :\n{documents[0].page_content[:300]}...")
print(f"📌 Metadata                   : {documents[0].metadata}")

📂 Number of documents loaded : 1
📝 Content preview            :
LangChain Architecture Overview

LangChain is an open-source framework developed to simplify building applications
powered by large language models (LLMs). It provides a modular set of components
that developers can combine to create sophisticated LLM pipelines.

Core Components:
1. Models: Wrappers...
📌 Metadata                   : {'source': 'sample_doc.txt'}


In [41]:
# ============================================================
# Text Splitter — Split large docs into smaller chunks
# Why? LLMs have token limits; smaller chunks = better retrieval
# ============================================================

# Install the specific package for text splitters if not present
!pip install -q langchain-text-splitters

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size    = 200,   # Max characters per chunk
    chunk_overlap = 30,    # Overlap keeps context between chunks
    length_function = len
)

chunks = splitter.split_documents(documents)

print(f"📊 Original doc size : {len(documents[0].page_content)} characters")
print(f"📊 Number of chunks  : {len(chunks)}")
print(f"\n📝 Chunk 1:\n{chunks[0].page_content}")
print(f"\n📝 Chunk 2:\n{chunks[1].page_content}")

📊 Original doc size : 812 characters
📊 Number of chunks  : 7

📝 Chunk 1:
LangChain Architecture Overview

📝 Chunk 2:
LangChain is an open-source framework developed to simplify building applications
powered by large language models (LLMs). It provides a modular set of components


In [42]:
# ============================================================
# WebBaseLoader — Load content from a website
# ============================================================

!pip install -q beautifulsoup4

from langchain_community.document_loaders import WebBaseLoader

# Load content from Wikipedia
web_loader = WebBaseLoader("https://en.wikipedia.org/wiki/LangChain")
web_docs   = web_loader.load()

print(f"📂 Pages loaded    : {len(web_docs)}")
print(f"📝 Content preview :\n{web_docs[0].page_content[:500]}...")

📂 Pages loaded    : 1
📝 Content preview :




LangChain - Wikipedia



























Jump to content







Main menu





Main menu
move to sidebar
hide



		Navigation
	


Main pageContentsCurrent eventsRandom articleAbout WikipediaContact us





		Contribute
	


HelpLearn to editCommunity portalRecent changesUpload fileSpecial pages



















Search











Search






















Appearance
















Donate

Create account

Log in








Personal tools





 Donate Create account Log in











...


---
## 🗃️ Section 8 — Vector Stores (Indexes) <a name='vectorstores'></a>

### Concept
**Vector Stores** convert text into **numerical embeddings** (vectors) and store them.
When you search, your query is also converted to a vector, and the store finds
the most **semantically similar** documents — even if they don't share exact keywords.

```
Text → Embedding Model → Vector [0.23, -0.51, 0.88, ...]
Query → Embedding → Compare vectors → Return closest matches
```

In [51]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

knowledge_base = [
    "LangChain is a framework for building LLM-powered applications.",
    "Chains in LangChain connect prompts, models, and output parsers in a pipeline.",
    "Agents in LangChain use LLMs to decide which tools to call dynamically.",
    "Memory allows LangChain applications to remember previous conversation turns.",
    "Vector stores enable semantic search over large document collections.",
    "RAG stands for Retrieval-Augmented Generation and improves LLM accuracy."
]

# FIXED: Using the active 2026 embedding model
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

vectorstore = FAISS.from_texts(knowledge_base, embeddings)

print("✅ Vector store built with Gemini Embeddings!")

✅ Vector store built with Gemini Embeddings!


In [52]:
# ============================================================
# Semantic Search — Find most relevant chunks for a query
# ============================================================

# Query in natural language
query = "How does an agent decide what to do?"

# k=3 returns top 3 most similar documents
results = vectorstore.similarity_search(query, k=3)

print(f"🔍 Query: {query}\n")
print("📄 Top Matching Documents:")
for i, doc in enumerate(results, 1):
    print(f"  {i}. {doc.page_content}")

🔍 Query: How does an agent decide what to do?

📄 Top Matching Documents:
  1. Agents in LangChain use LLMs to decide which tools to call dynamically.
  2. Chains in LangChain connect prompts, models, and output parsers in a pipeline.
  3. LangChain is a framework for building LLM-powered applications.


In [53]:
# ============================================================
# Save and Load Vector Store (for reuse without re-embedding)
# ============================================================

# Save to disk
vectorstore.save_local("faiss_index")
print("💾 Vector store saved to disk")

# Load back from disk
loaded_store = FAISS.load_local(
    "faiss_index",
    embeddings,
    allow_dangerous_deserialization=True
)
print("✅ Vector store loaded from disk")

# Test it works
test_results = loaded_store.similarity_search("What is RAG?", k=2)
print("\n🔍 Test search result:")
for doc in test_results:
    print(f"  → {doc.page_content}")

💾 Vector store saved to disk
✅ Vector store loaded from disk

🔍 Test search result:
  → RAG stands for Retrieval-Augmented Generation and improves LLM accuracy.
  → Agents in LangChain use LLMs to decide which tools to call dynamically.


---
## 🚀 Section 9 — End-to-End RAG Pipeline <a name='rag'></a>

### Concept
**RAG (Retrieval-Augmented Generation)** is the most common real-world LangChain
use case. Instead of relying only on the LLM's training data, RAG:
1. **Retrieves** relevant context from your document store
2. **Augments** the prompt with that context
3. **Generates** an answer grounded in your actual documents

```
User Question
     ↓
Vector Store → Retrieve top-k chunks
     ↓
Prompt = "Answer using this context: {chunks} \n Question: {question}"
     ↓
LLM → Grounded Answer
```

In [57]:
# Ensure all modern packages are installed
# !pip install -q -U langchain-core langchain-google-genai langchain-community langchain-text-splitters faiss-cpu

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# FIXED: Import from the dedicated text splitters package
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Load and split
# (Make sure "sample_doc.txt" exists in your directory)
loader = TextLoader("sample_doc.txt")
docs = loader.load()

# The syntax for the splitter itself remains exactly the same
splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=30)
chunks = splitter.split_documents(docs)

# 2. Gemini Embeddings & Vector Store
# FIXED: Using the active 2026 embedding model
embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# 3. Prompt & Model
rag_prompt = ChatPromptTemplate.from_template("""
Answer the question using ONLY the provided context.
Context: {context}
Question: {question}
Answer:
""")

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 4. LCEL Chain Construction
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("✅ Gemini RAG chain built successfully!")

# ============================================================
# Execution Example
# ============================================================
# To test your pipeline, invoke it like this:
# response = rag_chain.invoke("What does the document say about X?")
# print(response)

✅ Gemini RAG chain built successfully!


In [58]:
# ── STEP 5: Ask questions grounded in the document ───────────

questions = [
    "What are the core components of LangChain?",
    "What does LangChain help developers do?",
    "What is a vector store used for?"
]

for q in questions:
    print(f"\n❓ Question: {q}")
    answer = rag_chain.invoke(q)
    print(f"💬 Answer  : {answer}")
    print("-" * 60)


❓ Question: What are the core components of LangChain?
💬 Answer  : The core components of LangChain are:
1. Models
2. Prompts
3. Chains
4. Memory
5. Agents
6. Tools
7. Document Loaders
------------------------------------------------------------

❓ Question: What does LangChain help developers do?
💬 Answer  : LangChain helps developers rapidly prototype and deploy LLM applications by abstracting the complexity of working with models, data, and tools. It simplifies building applications powered by large language models (LLMs) and allows them to combine a modular set of components to create sophisticated LLM pipelines.
------------------------------------------------------------

❓ Question: What is a vector store used for?
💬 Answer  : A vector store is used for semantic similarity search.
------------------------------------------------------------


---
## 📊 Architecture Summary

```
┌─────────────────────────────────────────────────────────────┐
│                    LangChain Architecture                   │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  User Input                                                 │
│      │                                                      │
│      ▼                                                      │
│  Prompt Template  ←─── Memory (conversation history)       │
│      │                                                      │
│      ▼                                                      │
│  LLM (GPT-3.5 / GPT-4 / HuggingFace)                      │
│      │                                                      │
│      ▼                                                      │
│  Is it an Agent?                                            │
│      │                                                      │
│   Yes ──► Tool Selection ──► Tool Execution ──► Back to LLM│
│      │                                                      │
│    No ──► Output Parser ──► Final Answer                    │
│                                                             │
│  RAG Path:                                                  │
│  User Query → Vector Store → Retrieved Docs → Prompt → LLM │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

---
## ✅ Notebook Complete!

| Section | Status |
|---|---|
| Basic LLM Call | ✅ |
| Prompt Templates | ✅ |
| Chains (LCEL) | ✅ |
| Memory | ✅ |
| Agents & Custom Tools | ✅ |
| Document Loaders | ✅ |
| Vector Stores | ✅ |
| RAG Pipeline | ✅ |

---
*For the full blog post, architecture diagram, and real-world use cases, refer to the Medium blog.*